In [1]:
#!/usr/bin/env python3
"""
╔══════════════════════════════════════════════════════════════════════════════════╗
║  MULTIMODAL-WIRELESS DATASET GENERATOR — SINGLE-FILE COLAB EDITION             ║
║                                                                                ║
║  Paper: "Multimodal-Wireless: A Large-Scale Dataset for Sensing and            ║
║          Communication" (arXiv:2511.03220)                                     ║
║  Authors: Tianhao Mao, Le Liang et al., Southeast University                   ║
║                                                                                ║
║  Run in Google Colab:                                                          ║
║      1) Upload this file or paste into a cell                                  ║
║      2) !python multimodal_wireless_generator.py                               ║
║      3) Dataset saved to ./mmw_output/                                         ║
║                                                                                ║
║  OOD Design Patterns:                                                          ║
║    • Strategy   — Swappable generators at every pipeline stage                 ║
║    • Factory    — SensorFactory, create_weather()                              ║
║    • Composite  — CompositeLabelGenerator merges label types                   ║
║    • Template   — Pipeline.run() fixed flow, pluggable stages                  ║
║    • Builder    — PipelineConfig with nested dataclasses                       ║
║                                                                                ║
║  Class Hierarchy:                                                              ║
║    Sensor (ABC) ──► LiDARSensor, RGBCameraSensor, DepthCameraSensor,           ║
║                     IMUSensor, RadarSensor                                     ║
║    Agent (ABC)  ──► CAV (receiver/user), RSU (transmitter/BS)                  ║
║    WeatherCondition (ABC) ──► SunnyWeather, RainyWeather, FoggyWeather         ║
║    ScenarioGenerator (ABC) ──► CARLAScenarioGenerator,                         ║
║                                SyntheticScenarioGenerator                      ║
║    SceneReconstructor (ABC) ──► BlenderReconstructor,                           ║
║                                 SyntheticSceneBuilder                          ║
║    ChannelGenerator (ABC) ──► SionnaChannelGenerator,                          ║
║                               AnalyticalChannelGenerator                       ║
║    LabelGenerator (ABC) ──► BeamLabelGenerator, BoundingBoxLabelGenerator,     ║
║                             ChannelStateLabelGenerator,                        ║
║                             CompositeLabelGenerator                            ║
║    Pipeline ──► MultimodalWirelessPipeline, BatchPipelineRunner                ║
╚══════════════════════════════════════════════════════════════════════════════════╝
"""

import os, sys, json, time, logging, copy
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
from typing import List, Dict, Any, Optional, Tuple
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("mmw")

# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 1: CONFIGURATION DATACLASSES (Paper Section IV-A)                    ║
# ║  Maps to three config categories: Simulation, Scenario, Communication         ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

class TownID(Enum):
    """CARLA towns used in the dataset (Table III)."""
    TOWN03 = "Town03"   # Urban, roundabouts, undulating terrain
    TOWN05 = "Town05"   # Urban, CBD, ring road, largest map
    TOWN07 = "Town07"   # Rural, corn fields, sparse infrastructure
    TOWN10 = "Town10"   # Urban, high-rise, modern downtown

class WeatherType(Enum):
    SUNNY = "sunny"
    RAINY = "rainy"
    FOGGY = "foggy"

class AntennaLayout(Enum):
    ULA = "ula"   # Uniform Linear Array (2D)
    UPA = "upa"   # Uniform Planar Array (3D)

# ── Sensor specs from Table II ──

@dataclass
class RGBCameraSpec:
    width: int = 160; height: int = 120; fov_deg: float = 110.0

@dataclass
class DepthCameraSpec:
    width: int = 160; height: int = 120; fov_deg: float = 110.0

@dataclass
class LiDARSpec:
    channels: int = 64; points_per_second: int = 6000; range_m: float = 120.0
    upper_fov_deg: float = 2.0; lower_fov_deg: float = -25.0

@dataclass
class IMUSpec:
    gyro_noise_mean: float = 0.001; gyro_noise_std: float = 0.002
    accel_noise_std: float = 0.1

@dataclass
class RadarSpec:
    points_per_second: int = 400; range_m: float = 100.0
    vertical_fov_deg: float = 30.0; horizontal_fov_deg: float = 110.0

# ── Top-level config containers ──

@dataclass
class SimulationConfig:
    """Section IV-A: Simulation Settings."""
    frame_rate_hz: int = 100            # 100 Hz (5G NR frame-aligned)
    frame_duration_s: float = 0.01      # 10 ms per frame
    scenario_duration_s: float = 2.0    # Adjustable (paper: 8-13s)
    random_seed: int = 42
    @property
    def num_frames(self) -> int:
        return int(self.scenario_duration_s * self.frame_rate_hz)

@dataclass
class ScenarioConfig:
    """Section IV-A: Scenario Settings."""
    town: TownID = TownID.TOWN03
    scenario_name: str = "roundabout"
    weather: WeatherType = WeatherType.SUNNY
    num_cavs: int = 3
    num_rsus: int = 1
    num_background_vehicles: int = 20
    x_spawn: Tuple[float, float, float] = (0., 0., 0.)
    description: str = ""

@dataclass
class SensorConfig:
    """Section IV-A: Sensor Settings (Table II)."""
    rgb_camera: RGBCameraSpec = field(default_factory=RGBCameraSpec)
    depth_camera: DepthCameraSpec = field(default_factory=DepthCameraSpec)
    lidar: LiDARSpec = field(default_factory=LiDARSpec)
    imu: IMUSpec = field(default_factory=IMUSpec)
    radar: RadarSpec = field(default_factory=RadarSpec)
    cav_num_rgb_cameras: int = 4
    rsu_has_depth_camera: bool = True
    rsu_has_radar: bool = True

@dataclass
class CommunicationConfig:
    """Table V: Sionna ray-tracing & channel parameters."""
    carrier_frequency_hz: float = 28e9          # 28 GHz mmWave
    secondary_frequency_hz: float = 4.9e9       # Sub-6 GHz option
    antenna_pattern: str = "dipole"
    ray_samples: int = 1_000_000
    max_reflection_order: int = 1               # First-order
    polarization: str = "V"
    subcarrier_spacing_hz: float = 120e3        # 120 kHz
    num_subcarriers: int = 256                  # (paper: 1024)
    tx_num_antennas_2d: int = 64                # BS ULA
    rx_num_antennas_2d: int = 16                # UE ULA
    tx_rows_3d: int = 8; tx_cols_3d: int = 8    # BS UPA
    rx_rows_3d: int = 8; rx_cols_3d: int = 8    # UE UPA
    codebook_size: int = 64                     # Q-DFT codebook
    @property
    def bandwidth_hz(self) -> float:
        return self.num_subcarriers * self.subcarrier_spacing_hz
    @property
    def tx_num_antennas_3d(self) -> int:
        return self.tx_rows_3d * self.tx_cols_3d
    @property
    def rx_num_antennas_3d(self) -> int:
        return self.rx_rows_3d * self.rx_cols_3d

@dataclass
class PipelineConfig:
    """Master configuration aggregating all sub-configs."""
    simulation: SimulationConfig = field(default_factory=SimulationConfig)
    scenario: ScenarioConfig = field(default_factory=ScenarioConfig)
    sensor: SensorConfig = field(default_factory=SensorConfig)
    communication: CommunicationConfig = field(default_factory=CommunicationConfig)
    output_dir: str = "./mmw_output"
    use_3d_arrays: bool = False
    sensor_save_interval: int = 10    # Save sensors every Nth frame

    def save(self, path: str):
        import dataclasses
        def _c(o):
            if isinstance(o, Enum): return o.value
            if dataclasses.is_dataclass(o): return dataclasses.asdict(o)
            return o
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, "w") as f:
            json.dump(dataclasses.asdict(self), f, indent=2, default=_c)

# ── 16 Scenario Library (Table III) ──

SCENARIO_LIBRARY: Dict[str, ScenarioConfig] = {
    "town03_roundabout":       ScenarioConfig(TownID.TOWN03, "roundabout",       description="Vehicles in a roundabout"),
    "town03_gas_station":      ScenarioConfig(TownID.TOWN03, "gas_station",      description="Rich reflection under gas station"),
    "town03_t_junction_slope": ScenarioConfig(TownID.TOWN03, "t_junction_slope", description="Vertical movement on slope"),
    "town03_crossroad_slope":  ScenarioConfig(TownID.TOWN03, "crossroad_slope",  description="Crossroad with slope"),
    "town03_5way_intersection":ScenarioConfig(TownID.TOWN03, "5way_intersection",description="5-way intersection"),
    "town05_dual_skybridges":  ScenarioConfig(TownID.TOWN05, "dual_skybridges",  description="Sky-bridges blocking LOS"),
    "town05_ring_road":        ScenarioConfig(TownID.TOWN05, "ring_road",        description="Elevated ring road"),
    "town05_t_junction_overpass":ScenarioConfig(TownID.TOWN05,"t_junction_overpass",num_cavs=4,description="Overpass reflections"),
    "town05_cbd_crossroad":    ScenarioConfig(TownID.TOWN05, "cbd_crossroad",    num_cavs=4, description="Glass facade CBD"),
    "town05_parking_lot":      ScenarioConfig(TownID.TOWN05, "parking_lot",      description="Parked car scatterers"),
    "town07_single_lane":      ScenarioConfig(TownID.TOWN07, "single_lane",      description="Rural brick buildings"),
    "town07_rural_crossroad":  ScenarioConfig(TownID.TOWN07, "rural_crossroad",  description="Open-space intersection"),
    "town10_urban_crossroad":  ScenarioConfig(TownID.TOWN10, "urban_crossroad",  description="Wide lanes, complex traffic"),
    "town10_curvy_road":       ScenarioConfig(TownID.TOWN10, "curvy_road",       description="Winding road, oncoming traffic"),
    "town10_h_shaped_road":    ScenarioConfig(TownID.TOWN10, "h_shaped_road",    description="U-turn layout"),
    "town10_wide_skybridge":   ScenarioConfig(TownID.TOWN10, "wide_skybridge",   description="Broad overpass LOS blockage"),
}

# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 2: RADIO MATERIALS (Table IV, Table VI — ITU-R P.527-5 at 28 GHz)   ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

@dataclass(frozen=True)
class RadioMaterial:
    name: str; relative_permittivity: float; conductivity: float

ITU_MARBLE   = RadioMaterial("itu_marble",   5.24, 0.0016)
ITU_CONCRETE = RadioMaterial("itu_concrete", 5.31, 0.0326)
ITU_GLASS    = RadioMaterial("itu_glass",    6.27, 0.0043)
ITU_BRICK    = RadioMaterial("itu_brick",    3.75, 0.038)
ITU_METAL    = RadioMaterial("itu_metal",    1.0,  1e7)
ITU_VERY_DRY_GROUND_28  = RadioMaterial("itu_very_dry_grnd_28",  2.5, 0.03)
ITU_MEDIUM_DRY_GROUND_28= RadioMaterial("itu_medium_dry_grnd_28",3.0, 0.4)
ITU_WET_GROUND_28       = RadioMaterial("itu_wet_grnd_28",       3.0, 2.5)

class MaterialAssigner:
    """Assigns ITU radio materials to scene objects based on town/weather context."""
    BUILDING_FACADE = {TownID.TOWN03: ITU_MARBLE, TownID.TOWN05: ITU_GLASS,
                       TownID.TOWN07: ITU_BRICK,  TownID.TOWN10: ITU_MARBLE}
    BUILDING_ROOFTOP = ITU_CONCRETE
    TRAFFIC = ITU_METAL
    GROUND = {WeatherType.SUNNY: ITU_VERY_DRY_GROUND_28,
              WeatherType.RAINY: ITU_WET_GROUND_28,
              WeatherType.FOGGY: ITU_WET_GROUND_28}

    @classmethod
    def get_scene_materials(cls, town: TownID, weather: WeatherType) -> Dict[str, RadioMaterial]:
        return {"building_facade": cls.BUILDING_FACADE.get(town, ITU_MARBLE),
                "building_rooftop": cls.BUILDING_ROOFTOP,
                "ground": cls.GROUND[weather], "traffic": cls.TRAFFIC}

    @classmethod
    def as_sionna_dict(cls, town: TownID, weather: WeatherType) -> Dict[str, Dict[str, float]]:
        return {n: {"relative_permittivity": m.relative_permittivity,
                    "conductivity": m.conductivity}
                for n, m in cls.get_scene_materials(town, weather).items()}


# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 3: SENSOR HIERARCHY (ABC → 5 concrete sensors + Factory)             ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

class Sensor(ABC):
    """Abstract sensor attached to an agent (CAV or RSU)."""
    def __init__(self, sensor_id: str, parent_agent_id: str):
        self.sensor_id = sensor_id
        self.parent_agent_id = parent_agent_id

    @abstractmethod
    def generate_reading(self, frame_idx: int, pos: np.ndarray,
                         rot: np.ndarray, rng: np.random.Generator) -> Dict[str, Any]: ...

    @abstractmethod
    def get_spec_dict(self) -> Dict[str, Any]: ...

    def __repr__(self):
        return f"{self.__class__.__name__}({self.sensor_id})"


class LiDARSensor(Sensor):
    """64-channel LiDAR (Table II): 120m range, 360° azimuth."""
    def __init__(self, sid, aid, spec=None):
        super().__init__(sid, aid); self.spec = spec or LiDARSpec()

    def generate_reading(self, frame_idx, pos, rot, rng):
        n = self.spec.points_per_second // 100
        az = rng.uniform(0, 2*np.pi, n)
        el = rng.uniform(np.radians(self.spec.lower_fov_deg),
                         np.radians(self.spec.upper_fov_deg), n)
        d = rng.uniform(1, self.spec.range_m, n)
        pts = np.stack([d*np.cos(el)*np.cos(az),
                        d*np.cos(el)*np.sin(az),
                        d*np.sin(el)], -1).astype(np.float32) + pos
        return {"point_cloud": pts,
                "intensity": rng.uniform(0, 1, n).astype(np.float32)}

    def get_spec_dict(self):
        return {"type": "lidar", "channels": self.spec.channels,
                "range_m": self.spec.range_m}


class RGBCameraSensor(Sensor):
    """RGB camera with 110° FOV (Table II)."""
    def __init__(self, sid, aid, spec=None, direction="front"):
        super().__init__(sid, aid)
        self.spec = spec or RGBCameraSpec(); self.direction = direction

    def generate_reading(self, frame_idx, pos, rot, rng):
        img = rng.integers(0, 256, (self.spec.height, self.spec.width, 3), dtype=np.uint8)
        return {"rgb_image": img, "meta": {"direction": self.direction}}

    def get_spec_dict(self):
        return {"type": "rgb", "res": [self.spec.width, self.spec.height],
                "direction": self.direction}


class DepthCameraSensor(Sensor):
    """Depth camera with 110° FOV (Table II)."""
    def __init__(self, sid, aid, spec=None):
        super().__init__(sid, aid); self.spec = spec or DepthCameraSpec()

    def generate_reading(self, frame_idx, pos, rot, rng):
        return {"depth_map": rng.uniform(0.1, 120,
                (self.spec.height, self.spec.width)).astype(np.float32)}

    def get_spec_dict(self):
        return {"type": "depth", "res": [self.spec.width, self.spec.height]}


class IMUSensor(Sensor):
    """IMU with configurable noise model (Table II)."""
    def __init__(self, sid, aid, spec=None):
        super().__init__(sid, aid); self.spec = spec or IMUSpec()

    def generate_reading(self, frame_idx, pos, rot, rng):
        return {"gyroscope": rng.normal(self.spec.gyro_noise_mean,
                                        self.spec.gyro_noise_std, 3).astype(np.float32),
                "accelerometer": (np.array([0,0,9.81]) +
                                  rng.normal(0, self.spec.accel_noise_std, 3)).astype(np.float32)}

    def get_spec_dict(self):
        return {"type": "imu"}


class RadarSensor(Sensor):
    """Automotive radar: 100m range, 110°×30° FOV (Table II)."""
    def __init__(self, sid, aid, spec=None):
        super().__init__(sid, aid); self.spec = spec or RadarSpec()

    def generate_reading(self, frame_idx, pos, rot, rng):
        n = self.spec.points_per_second // 100
        return {"range": rng.uniform(1, self.spec.range_m, n).astype(np.float32),
                "azimuth": rng.uniform(-np.radians(self.spec.horizontal_fov_deg/2),
                                        np.radians(self.spec.horizontal_fov_deg/2), n).astype(np.float32),
                "velocity": rng.uniform(-30, 30, n).astype(np.float32)}

    def get_spec_dict(self):
        return {"type": "radar", "range_m": self.spec.range_m}


class SensorFactory:
    """Factory Method: creates full sensor suites for CAVs and RSUs."""
    @staticmethod
    def create_cav_sensors(aid: str, cfg: SensorConfig) -> List[Sensor]:
        s = [RGBCameraSensor(f"rgb_{d}", aid, cfg.rgb_camera, d)
             for d in ["front", "back", "left", "right"]]
        s += [LiDARSensor("lidar", aid, cfg.lidar), IMUSensor("imu", aid, cfg.imu)]
        return s

    @staticmethod
    def create_rsu_sensors(aid: str, cfg: SensorConfig) -> List[Sensor]:
        s = [RGBCameraSensor("rgb_front", aid, cfg.rgb_camera, "front"),
             LiDARSensor("lidar", aid, cfg.lidar)]
        if cfg.rsu_has_depth_camera:
            s.append(DepthCameraSensor("depth", aid, cfg.depth_camera))
        if cfg.rsu_has_radar:
            s.append(RadarSensor("radar", aid, cfg.radar))
        return s


# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 4: AGENT HIERARCHY (ABC → CAV receiver, RSU transmitter)             ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

class Agent(ABC):
    """Abstract V2X participant owning sensors + antenna array + kinematics."""
    def __init__(self, agent_id: str, position: np.ndarray, rotation: np.ndarray):
        self.agent_id = agent_id
        self.position = np.asarray(position, dtype=np.float64)
        self.rotation = np.asarray(rotation, dtype=np.float64)
        self.sensors: List[Sensor] = []
        self.trajectory: List[np.ndarray] = [self.position.copy()]

    @abstractmethod
    def get_role(self) -> str:
        """Return 'tx' or 'rx' for channel link direction."""
        ...

    @abstractmethod
    def get_antenna_config(self, cc: CommunicationConfig, use_3d: bool) -> Dict[str, Any]: ...

    def attach_sensors(self, sl: List[Sensor]):
        self.sensors = sl

    def update_pose(self, p: np.ndarray, r: np.ndarray):
        self.position = np.asarray(p, dtype=np.float64)
        self.rotation = np.asarray(r, dtype=np.float64)
        self.trajectory.append(self.position.copy())

    def collect_sensor_data(self, fi: int, rng: np.random.Generator) -> Dict[str, Any]:
        return {s.sensor_id: s.generate_reading(fi, self.position, self.rotation, rng)
                for s in self.sensors}

    def get_pose_dict(self) -> Dict[str, Any]:
        return {"agent_id": self.agent_id, "position": self.position.tolist(),
                "rotation": self.rotation.tolist(), "role": self.get_role()}

    def __repr__(self):
        return f"{self.__class__.__name__}({self.agent_id}, pos={self.position.round(1)})"


class CAV(Agent):
    """Connected Autonomous Vehicle — mobile user (receiver in V2I downlink)."""
    def __init__(self, aid, pos, rot, velocity=10.0):
        super().__init__(aid, pos, rot); self.velocity = velocity

    def get_role(self): return "rx"

    def get_antenna_config(self, cc, use_3d):
        if use_3d: return {"layout": "upa", "antennas": cc.rx_num_antennas_3d}
        return {"layout": "ula", "antennas": cc.rx_num_antennas_2d}

    @classmethod
    def from_config(cls, idx, pos, rot, scfg):
        aid = f"cav_{idx:02d}"
        c = cls(aid, pos, rot)
        c.attach_sensors(SensorFactory.create_cav_sensors(aid, scfg))
        return c


class RSU(Agent):
    """Road-Side Unit — base station (transmitter in V2I downlink)."""
    def __init__(self, aid, pos, rot):
        super().__init__(aid, pos, rot)

    def get_role(self): return "tx"

    def get_antenna_config(self, cc, use_3d):
        if use_3d: return {"layout": "upa", "antennas": cc.tx_num_antennas_3d}
        return {"layout": "ula", "antennas": cc.tx_num_antennas_2d}

    @classmethod
    def from_config(cls, idx, pos, rot, scfg):
        aid = f"rsu_{idx:02d}"
        r = cls(aid, pos, rot)
        r.attach_sensors(SensorFactory.create_rsu_sensors(aid, scfg))
        return r


# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 5: WEATHER HIERARCHY (ABC → Sunny/Rainy/Foggy + sensor degradation)  ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

class WeatherCondition(ABC):
    """Abstract weather affecting sensors and channel propagation."""
    @abstractmethod
    def get_type(self) -> WeatherType: ...
    @abstractmethod
    def get_ground_material(self) -> str: ...
    @abstractmethod
    def degrade_lidar(self, pts, inten, rng) -> tuple: ...
    @abstractmethod
    def degrade_rgb(self, img, rng) -> np.ndarray: ...


class SunnyWeather(WeatherCondition):
    """Clear conditions — no sensor degradation."""
    def get_type(self): return WeatherType.SUNNY
    def get_ground_material(self): return "itu_very_dry_ground"
    def degrade_lidar(self, pts, i, rng): return pts, i
    def degrade_rgb(self, img, rng): return img


class RainyWeather(WeatherCondition):
    """Rain — LiDAR Mie scattering false echoes, camera noise/blur."""
    def __init__(self, intensity=0.6): self.rain_intensity = intensity
    def get_type(self): return WeatherType.RAINY
    def get_ground_material(self): return "itu_wet_ground"

    def degrade_lidar(self, pts, inten, rng):
        n_f = int(len(pts) * 0.05 * self.rain_intensity)
        if n_f > 0:
            fp = rng.uniform(-3, 3, (n_f, 3)).astype(np.float32) + \
                 pts[rng.integers(0, len(pts), n_f)]
            pts = np.concatenate([pts, fp])
            inten = np.concatenate([inten, rng.uniform(0, 0.3, n_f).astype(np.float32)])
        return pts, inten * (1 - 0.4 * self.rain_intensity)

    def degrade_rgb(self, img, rng):
        return np.clip(img.astype(np.float32) * 0.85 +
                       rng.normal(0, 15*self.rain_intensity, img.shape),
                       0, 255).astype(np.uint8)


class FoggyWeather(WeatherCondition):
    """Fog — exponential LiDAR attenuation, camera visibility reduction."""
    def __init__(self, density=0.5): self.fog_density = density
    def get_type(self): return WeatherType.FOGGY
    def get_ground_material(self): return "itu_wet_ground"

    def degrade_lidar(self, pts, inten, rng):
        d = np.linalg.norm(pts, axis=1)
        vr = 120 * (1 - self.fog_density * 0.8)
        mask = rng.random(len(pts)) < np.exp(-d / vr)
        return pts[mask], inten[mask]

    def degrade_rgb(self, img, rng):
        a = self.fog_density * 0.6
        return np.clip(img.astype(np.float32)*(1-a) + 180*a, 0, 255).astype(np.uint8)


def create_weather(wt: WeatherType, **kw) -> WeatherCondition:
    """Factory function for weather conditions."""
    return {WeatherType.SUNNY: SunnyWeather, WeatherType.RAINY: RainyWeather,
            WeatherType.FOGGY: FoggyWeather}[wt](**kw)


# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 6: SCENARIO GENERATION (ABC → CARLA / Synthetic)                     ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

@dataclass
class FrameData:
    """Immutable snapshot of one simulation frame."""
    frame_idx: int
    timestamp_s: float
    agent_poses: Dict[str, Dict[str, Any]]
    sensor_readings: Dict[str, Dict[str, Any]]
    bounding_boxes: List[Dict[str, Any]]


class ScenarioGenerator(ABC):
    """Abstract base for Stage 1: scenario execution & data capture."""
    def __init__(self, config: PipelineConfig, weather: WeatherCondition):
        self.config = config; self.weather = weather
        self.agents: List[Agent] = []

    @abstractmethod
    def setup(self): ...
    @abstractmethod
    def step(self, frame_idx: int) -> FrameData: ...
    @abstractmethod
    def teardown(self): ...
    def get_agents(self) -> List[Agent]: return self.agents


class CARLAScenarioGenerator(ScenarioGenerator):
    """Connects to live CARLA server. Requires: pip install carla==0.9.15."""
    def __init__(self, config, weather, host="localhost", port=2000):
        super().__init__(config, weather)
        self.host = host; self.port = port

    def setup(self):
        try: import carla
        except ImportError:
            raise RuntimeError("CARLA not installed. Use SyntheticScenarioGenerator.")
        logger.info(f"CARLA generator: connecting to {self.host}:{self.port}")

    def step(self, frame_idx):
        raise NotImplementedError("Requires running CARLA server")

    def teardown(self): pass


class SyntheticScenarioGenerator(ScenarioGenerator):
    """Pure-NumPy synthetic V2X trajectories. Runs anywhere (no GPU needed)."""
    def __init__(self, config, weather):
        super().__init__(config, weather)
        self.rng = np.random.default_rng(config.simulation.random_seed)

    def setup(self):
        sc = self.config.scenario
        # RSU: elevated static position (typical roadside infrastructure)
        self.agents.append(RSU.from_config(0, np.array([0., 0., 6.]),
                                            np.zeros(3), self.config.sensor))
        x_spawn = np.array(sc.x_spawn)
        for i in range(sc.num_cavs):
            a = 2*np.pi*i/sc.num_cavs + self.rng.uniform(-0.3, 0.3)
            r = self.rng.uniform(20, 60)
            pos = x_spawn + np.array([r*np.cos(a), r*np.sin(a), 0.3])
            rot = np.array([0., 0., np.degrees(a)+180+self.rng.uniform(-15, 15)])
            cav = CAV.from_config(i, pos, rot, self.config.sensor)
            cav.velocity = self.rng.uniform(5, 15)
            self.agents.append(cav)
        logger.info(f"Synthetic: {sc.num_cavs} CAVs + 1 RSU, "
                    f"weather={self.weather.get_type().value}")

    def step(self, frame_idx):
        dt = self.config.simulation.frame_duration_s
        rng = np.random.default_rng(self.config.simulation.random_seed + frame_idx)

        # Advance CAV kinematics (constant speed + small steering noise)
        for a in self.agents:
            if isinstance(a, CAV):
                yr = np.radians(a.rotation[2]) + self.rng.normal(0, 0.02)
                new_pos = a.position + np.array([
                    a.velocity*dt*np.cos(yr), a.velocity*dt*np.sin(yr), 0.])
                a.update_pose(new_pos, np.array([0., 0., np.degrees(yr)]))

        poses, readings, bboxes = {}, {}, []
        for a in self.agents:
            poses[a.agent_id] = a.get_pose_dict()
            rd = a.collect_sensor_data(frame_idx, rng)
            # Apply weather degradation polymorphically
            for sid, d in rd.items():
                if "point_cloud" in d:
                    d["point_cloud"], d["intensity"] = \
                        self.weather.degrade_lidar(d["point_cloud"], d["intensity"], rng)
                if "rgb_image" in d:
                    d["rgb_image"] = self.weather.degrade_rgb(d["rgb_image"], rng)
            readings[a.agent_id] = rd
            if isinstance(a, CAV):
                bboxes.append({"agent_id": a.agent_id,
                               "position": a.position.tolist(),
                               "extent": [2.5, 1.0, 0.8],
                               "rotation": a.rotation.tolist(),
                               "velocity": a.velocity})
        return FrameData(frame_idx, frame_idx*dt, poses, readings, bboxes)

    def teardown(self):
        logger.info("Synthetic scenario complete")


# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 7: SCENE RECONSTRUCTION (ABC → Blender / SyntheticSceneBuilder)      ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

@dataclass
class SceneObject:
    """Geometric object in the ray-tracing scene."""
    obj_id: str; obj_type: str; position: np.ndarray; rotation: np.ndarray
    dimensions: np.ndarray; material_name: str


@dataclass
class SceneDescription:
    """Complete scene for one frame, ready for Sionna channel computation."""
    frame_idx: int; timestamp_s: float
    static_objects: List[SceneObject]; dynamic_objects: List[SceneObject]
    tx_positions: Dict[str, np.ndarray]; rx_positions: Dict[str, np.ndarray]
    materials: Dict[str, Dict[str, float]]; ground_material: str

    def get_all_objects(self) -> List[SceneObject]:
        return self.static_objects + self.dynamic_objects


class SceneReconstructor(ABC):
    """Abstract base for Stage 2: scene reconstruction."""
    def __init__(self, config: PipelineConfig):
        self.config = config
        self.materials = MaterialAssigner.as_sionna_dict(
            config.scenario.town, config.scenario.weather)

    @abstractmethod
    def build_static_environment(self) -> List[SceneObject]: ...
    @abstractmethod
    def place_dynamic_actors(self, frame: FrameData) -> List[SceneObject]: ...

    def reconstruct_frame(self, frame: FrameData) -> SceneDescription:
        static = self.build_static_environment()
        dynamic = self.place_dynamic_actors(frame)
        tx, rx = {}, {}
        for aid, p in frame.agent_poses.items():
            pos = np.array(p["position"])
            (tx if p["role"] == "tx" else rx)[aid] = pos
        gm = MaterialAssigner.get_scene_materials(
            self.config.scenario.town, self.config.scenario.weather)["ground"]
        return SceneDescription(frame.frame_idx, frame.timestamp_s,
                                static, dynamic, tx, rx,
                                self.materials, gm.name)


class BlenderReconstructor(SceneReconstructor):
    """Headless Blender scene reconstruction. Requires Blender 3.6.x binary."""
    def __init__(self, config, town_blend_path=""):
        super().__init__(config); self.town_blend_path = town_blend_path

    def build_static_environment(self):
        mat = MaterialAssigner.get_scene_materials(
            self.config.scenario.town, self.config.scenario.weather)
        return [SceneObject("ground", "ground", np.zeros(3), np.zeros(3),
                            np.array([400, 400, 0.1]), mat["ground"].name)]

    def place_dynamic_actors(self, frame):
        mat = MaterialAssigner.get_scene_materials(
            self.config.scenario.town, self.config.scenario.weather)
        return [SceneObject(b["agent_id"], "vehicle", np.array(b["position"]),
                np.array(b["rotation"]), np.array(b["extent"])*2,
                mat["traffic"].name) for b in frame.bounding_boxes]


class SyntheticSceneBuilder(SceneReconstructor):
    """Builds ray-tracing scenes in pure Python (no Blender needed)."""
    def __init__(self, config):
        super().__init__(config); self._cache = None

    def build_static_environment(self):
        if self._cache is not None: return self._cache
        town = self.config.scenario.town
        mat = MaterialAssigner.get_scene_materials(town, self.config.scenario.weather)
        rng = np.random.default_rng(abs(hash(town.value)) % (2**31))
        objs = [SceneObject("ground", "ground", np.zeros(3), np.zeros(3),
                            np.array([400, 400, 0.1]), mat["ground"].name)]
        density = {"Town03": 0.6, "Town05": 0.8,
                   "Town07": 0.2, "Town10": 0.9}[town.value]
        for i in range(int(40 * density)):
            side = rng.choice([-1, 1]); along = rng.uniform(-150, 150)
            offset = side * rng.uniform(15, 30)
            h = rng.uniform(8, 40*density)
            w, d = rng.uniform(10, 25), rng.uniform(10, 25)
            objs.append(SceneObject(
                f"bldg_{i:03d}", "building",
                np.array([along, offset, h/2]), np.zeros(3),
                np.array([w, d, h]), mat["building_facade"].name))
        self._cache = objs
        return objs

    def place_dynamic_actors(self, frame):
        mat = MaterialAssigner.get_scene_materials(
            self.config.scenario.town, self.config.scenario.weather)
        return [SceneObject(b["agent_id"], "vehicle", np.array(b["position"]),
                np.array(b["rotation"]), np.array(b["extent"])*2,
                mat["traffic"].name) for b in frame.bounding_boxes]


# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 8: CHANNEL GENERATION — Eq. (1)-(4) from the paper                   ║
# ║  ChannelGenerator (ABC) → Sionna / Analytical                                 ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

@dataclass
class PathData:
    """Raw physics for one propagation path: {A_m, τ_m, AOD, AOA}."""
    path_idx: int; delay_s: float; complex_gain: np.ndarray  # [Nr, Nt]
    aod_azimuth_rad: float; aod_zenith_rad: float
    aoa_azimuth_rad: float; aoa_zenith_rad: float; is_los: bool


@dataclass
class ChannelRealization:
    """Complete channel for one TX→RX link at one frame."""
    frame_idx: int; tx_id: str; rx_id: str
    tx_position: np.ndarray; rx_position: np.ndarray
    distance_m: float; num_paths: int; paths: List[PathData]
    _H_freq: Optional[np.ndarray] = field(default=None, repr=False)
    _H_avg: Optional[np.ndarray] = field(default=None, repr=False)

    def get_frequency_response(self, cc: CommunicationConfig) -> np.ndarray:
        """Eq.(1): H(f_k) = Σ_m A_m · exp(-j2πf_k·τ_m).  Returns [Nr, Nt, K]."""
        if self._H_freq is not None: return self._H_freq
        K = cc.num_subcarriers; df = cc.subcarrier_spacing_hz
        f_k = (np.arange(K) - (K+1)/2) * df   # baseband subcarrier frequencies
        Nr, Nt = self.paths[0].complex_gain.shape if self.paths else (1, 1)
        H = np.zeros((Nr, Nt, K), dtype=np.complex64)
        for p in self.paths:
            phase = np.exp(-1j * 2*np.pi * f_k * p.delay_s).astype(np.complex64)
            H += p.complex_gain[:, :, None] * phase[None, None, :]
        self._H_freq = H
        return H

    def get_avg_channel(self, cc: CommunicationConfig) -> np.ndarray:
        """H̄_n = mean_k H(f_k). Used in Eq.(3)-(4)."""
        if self._H_avg is not None: return self._H_avg
        self._H_avg = self.get_frequency_response(cc).mean(axis=2)
        return self._H_avg


def dft_codebook(N: int, Q: int) -> np.ndarray:
    """Eq.(2): Q-DFT codebook.
    f(q) = (1/√N) [1, e^{j2πq/Q}, ..., e^{j2π(N-1)q/Q}]^T.
    Returns [Q, N]."""
    q = np.arange(Q)[:, None]; n = np.arange(N)[None, :]
    return (1.0 / np.sqrt(N)) * np.exp(1j * 2*np.pi * q * n / Q).astype(np.complex64)


def compute_optimal_beam(H: np.ndarray, F_tx: np.ndarray,
                         F_rx: np.ndarray) -> Tuple[int, int, float]:
    """Eq.(4): (p*, q*) = argmax |w(p)^H · H · f(q)|².
    Returns (p*, q*, max_gain)."""
    gains = np.abs(F_rx.conj() @ H @ F_tx.T) ** 2   # [Q_rx, Q_tx]
    idx = np.unravel_index(np.argmax(gains), gains.shape)
    return int(idx[0]), int(idx[1]), float(gains[idx])


def compute_normalized_gain(H, q_hat, p_star, F_tx, F_rx, max_gain) -> float:
    """Eq.(3): G(q̂) = |w(p*)^H H f(q̂)|² / |w(p*)^H H f(q*)|²."""
    if max_gain <= 0: return 0.0
    return float(np.abs(F_rx[p_star].conj() @ H @ F_tx[q_hat])**2 / max_gain)


class ChannelGenerator(ABC):
    """Abstract base for Stage 3: channel generation."""
    def __init__(self, config: PipelineConfig):
        self.config = config; self.comm = config.communication

    @abstractmethod
    def generate_channel(self, scene: SceneDescription,
                         tx_id: str, rx_id: str) -> ChannelRealization: ...

    def generate_all_links(self, scene: SceneDescription) -> List[ChannelRealization]:
        return [self.generate_channel(scene, tx, rx)
                for tx in scene.tx_positions for rx in scene.rx_positions]


class SionnaChannelGenerator(ChannelGenerator):
    """Full Sionna RT ray-tracing. Requires: pip install sionna==0.19.2 + GPU."""
    def __init__(self, config, scene_xml_dir=None):
        super().__init__(config); self.scene_xml_dir = scene_xml_dir
        try:
            import sionna; logger.info(f"Sionna v{sionna.__version__}")
        except ImportError:
            raise ImportError("Sionna not installed")

    def generate_channel(self, scene, tx_id, rx_id):
        raise NotImplementedError("Requires Sionna + exported scene XMLs")


class AnalyticalChannelGenerator(ChannelGenerator):
    """Geometric channel model: LOS + first-order reflections (no GPU needed).
    Uses ULA steering vectors, free-space path loss, and Fresnel reflection."""
    def __init__(self, config):
        super().__init__(config)
        self.rng = np.random.default_rng(config.simulation.random_seed)
        self.c = 3e8
        self.wl = self.c / config.communication.carrier_frequency_hz

    def _steering(self, N, angle):
        """ULA steering vector a(θ) = [1, e^{jπsinθ}, ...]."""
        return np.exp(1j * np.pi * np.arange(N) * np.sin(angle)).astype(np.complex64)

    def _fspl(self, d):
        """Free-space path loss at carrier frequency."""
        return self.wl / (4 * np.pi * max(d, 0.1))

    def generate_channel(self, scene, tx_id, rx_id):
        tx = scene.tx_positions[tx_id]; rx = scene.rx_positions[rx_id]
        dist = float(np.linalg.norm(tx - rx))
        u3d = self.config.use_3d_arrays
        Nt = self.comm.tx_num_antennas_3d if u3d else self.comm.tx_num_antennas_2d
        Nr = self.comm.rx_num_antennas_3d if u3d else self.comm.rx_num_antennas_2d
        paths = []
        d = rx - tx
        aod = np.arctan2(d[1], d[0]); aoa = np.arctan2(-d[1], -d[0])

        # ── Check LOS blockage by buildings ──
        los_blocked = False
        for obj in scene.static_objects:
            if obj.obj_type == "building":
                cross = d[0]*(obj.position[1]-tx[1]) - d[1]*(obj.position[0]-tx[0])
                dot = np.dot((obj.position - tx)[:2], d[:2])
                if abs(cross) < obj.dimensions[0] and 0 < dot < np.dot(d[:2], d[:2]):
                    los_blocked = True; break

        if not los_blocked:
            g = self._fspl(dist)
            A = g * np.outer(self._steering(Nr, aoa), self._steering(Nt, aod).conj())
            paths.append(PathData(0, dist/self.c, A, aod, 0, aoa, 0, True))

        # ── First-order reflections from buildings + vehicles ──
        for obj in scene.get_all_objects()[:8]:
            if obj.obj_type not in ("building", "vehicle"): continue
            rp = obj.position + self.rng.uniform(-0.5, 0.5, 3) * obj.dimensions * 0.5
            d1 = float(np.linalg.norm(tx - rp))
            d2 = float(np.linalg.norm(rp - rx))
            td = d1 + d2
            if td > 300: continue

            eps = scene.materials.get(obj.material_name, {}).get(
                "relative_permittivity", 5.0)
            ti = self.rng.uniform(0.1, np.pi/2)
            ct = np.sqrt(max(0, 1 - (np.sin(ti)**2) / eps))
            rc = (np.cos(ti) - np.sqrt(eps)*ct) / (np.cos(ti) + np.sqrt(eps)*ct)
            g = self._fspl(td) * abs(rc)
            a1 = np.arctan2((rp-tx)[1], (rp-tx)[0])
            a2 = np.arctan2((rx-rp)[1], (rx-rp)[0])
            A = g * np.exp(1j*self.rng.uniform(0, 2*np.pi)) * \
                np.outer(self._steering(Nr, a2), self._steering(Nt, a1).conj())
            paths.append(PathData(len(paths), td/self.c, A, a1, 0, a2, 0, False))

        if not paths:   # Guarantee at least one weak NLOS path
            A = 1e-6 * np.outer(
                self._steering(Nr, self.rng.uniform(-np.pi, np.pi)),
                self._steering(Nt, self.rng.uniform(-np.pi, np.pi)).conj())
            paths.append(PathData(0, dist/self.c*1.5, A, 0, 0, 0, 0, False))

        return ChannelRealization(scene.frame_idx, tx_id, rx_id, tx, rx,
                                  dist, len(paths), paths)


# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 9: LABEL GENERATORS — Eq. (2)-(4) beam labels + bounding boxes       ║
# ║  LabelGenerator (ABC) → Beam / BBox / CSI / Composite                         ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

@dataclass
class FrameLabels:
    """All labels for one frame, keyed by label type."""
    frame_idx: int; labels: Dict[str, Any]

    def merge(self, other: "FrameLabels"):
        assert self.frame_idx == other.frame_idx
        self.labels.update(other.labels)

    def to_dict(self) -> Dict[str, Any]:
        def _conv(v):
            if isinstance(v, np.ndarray): return v.tolist()
            if isinstance(v, (np.integer,)): return int(v)
            if isinstance(v, (np.floating,)): return float(v)
            if isinstance(v, dict): return {k: _conv(w) for k, w in v.items()}
            if isinstance(v, list): return [_conv(w) for w in v]
            return v
        return _conv({"frame_idx": self.frame_idx, **self.labels})


class LabelGenerator(ABC):
    """Abstract label generator."""
    def __init__(self, config: PipelineConfig):
        self.config = config

    @abstractmethod
    def generate_frame_labels(self, frame: FrameData,
                              channels: Optional[List[ChannelRealization]] = None
                              ) -> FrameLabels: ...


class BeamLabelGenerator(LabelGenerator):
    """Optimal beam indices + normalized gain via Eq. (2)-(4)."""
    def __init__(self, config):
        super().__init__(config)
        comm = config.communication
        Nt = comm.tx_num_antennas_3d if config.use_3d_arrays else comm.tx_num_antennas_2d
        Nr = comm.rx_num_antennas_3d if config.use_3d_arrays else comm.rx_num_antennas_2d
        Q = comm.codebook_size
        self.F_tx = dft_codebook(Nt, Q)
        self.F_rx = dft_codebook(Nr, Q)
        self.Nt, self.Nr, self.Q = Nt, Nr, Q

    def generate_frame_labels(self, frame, channels=None):
        if not channels: return FrameLabels(frame.frame_idx, {"beam": {}})
        beam_labels = {}
        for ch in channels:
            link = f"{ch.tx_id}_to_{ch.rx_id}"
            H_avg = ch.get_avg_channel(self.config.communication)
            p_star, q_star, max_gain = compute_optimal_beam(H_avg, self.F_tx, self.F_rx)
            H_freq = ch.get_frequency_response(self.config.communication)
            ch_power = float(np.mean(np.abs(H_freq)**2))
            beam_labels[link] = {
                "optimal_tx_beam_idx": q_star,
                "optimal_rx_beam_idx": p_star,
                "max_beamforming_gain": max_gain,
                "channel_power_dB": float(10*np.log10(ch_power + 1e-30)),
                "distance_m": ch.distance_m,
                "num_paths": ch.num_paths,
                "is_los": any(p.is_los for p in ch.paths),
                "tx_position": ch.tx_position.tolist(),
                "rx_position": ch.rx_position.tolist(),
                "path_delays_s": [p.delay_s for p in ch.paths],
                "path_aod_az_rad": [p.aod_azimuth_rad for p in ch.paths],
                "path_aoa_az_rad": [p.aoa_azimuth_rad for p in ch.paths],
            }
        return FrameLabels(frame.frame_idx, {"beam": beam_labels})


class BoundingBoxLabelGenerator(LabelGenerator):
    """3D bounding box labels for perception tasks."""
    def generate_frame_labels(self, frame, channels=None):
        boxes = [{"agent_id": b["agent_id"], "center": b["position"],
                  "extent": b["extent"], "rotation": b["rotation"],
                  "velocity": b.get("velocity", 0.), "class": "vehicle"}
                 for b in frame.bounding_boxes]
        return FrameLabels(frame.frame_idx, {"bounding_boxes": boxes})


class ChannelStateLabelGenerator(LabelGenerator):
    """Full CSI labels: H(f_k) matrices per subcarrier."""
    def generate_frame_labels(self, frame, channels=None):
        if not channels: return FrameLabels(frame.frame_idx, {"csi": {}})
        csi = {}
        for ch in channels:
            link = f"{ch.tx_id}_to_{ch.rx_id}"
            H = ch.get_frequency_response(self.config.communication)
            csi[link] = {"H_shape": list(H.shape)}  # Full H saved in .npz
        return FrameLabels(frame.frame_idx, {"csi": csi})


class CompositeLabelGenerator(LabelGenerator):
    """Composite pattern: merges outputs from multiple label generators."""
    def __init__(self, config, generators=None):
        super().__init__(config)
        self.generators: List[LabelGenerator] = generators or []

    def add(self, gen: LabelGenerator) -> "CompositeLabelGenerator":
        self.generators.append(gen); return self  # fluent API

    def generate_frame_labels(self, frame, channels=None):
        merged = FrameLabels(frame.frame_idx, {})
        for g in self.generators:
            merged.merge(g.generate_frame_labels(frame, channels))
        return merged


# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 10: DATASET I/O                                                      ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

class DatasetWriter:
    """Writes dataset to organized directory structure."""
    def __init__(self, output_dir):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

    @staticmethod
    def _to_json_safe(obj):
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, (np.integer,)): return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, dict): return {k: DatasetWriter._to_json_safe(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)): return [DatasetWriter._to_json_safe(v) for v in obj]
        return obj

    def get_frame_dir(self, scenario, weather, frame_idx):
        d = os.path.join(self.output_dir, scenario, weather,
                         "frames", f"frame_{frame_idx:06d}")
        os.makedirs(d, exist_ok=True); return d

    def write_json(self, path, data):
        with open(path, "w") as f:
            json.dump(self._to_json_safe(data), f, indent=2)

    def write_sensor_data(self, fdir, agent_id, sensor_id, data):
        sd = os.path.join(fdir, agent_id, sensor_id)
        os.makedirs(sd, exist_ok=True)
        for k, v in data.items():
            if isinstance(v, np.ndarray):
                np.save(os.path.join(sd, f"{k}.npy"), v)
            elif isinstance(v, dict):
                self.write_json(os.path.join(sd, f"{k}.json"), v)

    def write_channel(self, fdir, tx_id, rx_id, H, metadata):
        cd = os.path.join(fdir, "channel"); os.makedirs(cd, exist_ok=True)
        link = f"{tx_id}_to_{rx_id}"
        np.savez_compressed(os.path.join(cd, f"{link}.npz"),
                            H_real=np.real(H).astype(np.float32),
                            H_imag=np.imag(H).astype(np.float32))
        self.write_json(os.path.join(cd, f"{link}_meta.json"), metadata)


class DatasetReader:
    """Reads generated dataset for downstream tasks."""
    def __init__(self, dataset_dir):
        self.dataset_dir = dataset_dir

    def load_frame_labels(self, scenario, weather, frame_idx):
        p = os.path.join(self.dataset_dir, scenario, weather,
                         "frames", f"frame_{frame_idx:06d}", "labels.json")
        with open(p) as f: return json.load(f)

    def load_channel(self, scenario, weather, frame_idx, link):
        cd = os.path.join(self.dataset_dir, scenario, weather,
                          "frames", f"frame_{frame_idx:06d}", "channel")
        d = np.load(os.path.join(cd, f"{link}.npz"))
        return d["H_real"] + 1j * d["H_imag"]


# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 11: PIPELINE ORCHESTRATOR (Template Method pattern)                   ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

class MultimodalWirelessPipeline:
    """End-to-end orchestrator: Configure → Scenario → Reconstruct → Channel → Labels → Save.

    Auto-selects available backends:
        CARLA → Synthetic,  Blender → SyntheticScene,  Sionna → Analytical
    """
    def __init__(self, config: PipelineConfig,
                 scenario_gen=None, scene_recon=None,
                 channel_gen=None, label_gen=None):
        self.config = config
        self.writer = DatasetWriter(config.output_dir)
        self.weather = create_weather(config.scenario.weather)

        # Auto-detect backends (Strategy pattern)
        self.scenario_gen = scenario_gen or self._auto_scenario()
        self.scene_recon = scene_recon or SyntheticSceneBuilder(config)
        self.channel_gen = channel_gen or self._auto_channel()
        self.label_gen = label_gen or (
            CompositeLabelGenerator(config)
            .add(BeamLabelGenerator(config))
            .add(BoundingBoxLabelGenerator(config))
        )

    def _auto_scenario(self):
        try:
            g = CARLAScenarioGenerator(self.config, self.weather); import carla; return g
        except Exception:
            return SyntheticScenarioGenerator(self.config, self.weather)

    def _auto_channel(self):
        try: return SionnaChannelGenerator(self.config)
        except Exception: return AnalyticalChannelGenerator(self.config)

    def run(self) -> Dict[str, Any]:
        """Template Method: fixed pipeline flow with pluggable stages."""
        t0 = time.time()
        sc_name = self.config.scenario.scenario_name
        w_name = self.config.scenario.weather.value
        n_frames = self.config.simulation.num_frames

        logger.info("=" * 65)
        logger.info("MULTIMODAL-WIRELESS DATASET GENERATION")
        logger.info(f"  Scenario : {sc_name} | Town: {self.config.scenario.town.value}")
        logger.info(f"  Weather  : {w_name} | Frames: {n_frames} @ "
                    f"{self.config.simulation.frame_rate_hz} Hz")
        logger.info(f"  Antennas : Nt={self.config.communication.tx_num_antennas_2d}, "
                    f"Nr={self.config.communication.rx_num_antennas_2d}, "
                    f"Q={self.config.communication.codebook_size}")
        logger.info("=" * 65)

        # Stage 1: Scenario
        logger.info("\n STAGE 1: Scenario Execution & Data Capture")
        self.scenario_gen.setup()
        frames = []
        for i in range(n_frames):
            frames.append(self.scenario_gen.step(i))
            if (i+1) % max(1, n_frames//5) == 0:
                logger.info(f"  Frame {i+1}/{n_frames}")
        self.scenario_gen.teardown()
        agents = self.scenario_gen.get_agents()
        logger.info(f"   {len(frames)} frames, {len(agents)} agents")

        # Stage 2+3+4: Process each frame (streaming to control memory)
        logger.info("\n STAGES 2-4: Reconstruct → Channel → Labels → Save")
        total_channels = 0
        for i, frame in enumerate(frames):
            # Reconstruct scene
            scene = self.scene_recon.reconstruct_frame(frame)
            # Generate channels
            channels = self.channel_gen.generate_all_links(scene)
            total_channels += len(channels)
            # Generate labels
            labels = self.label_gen.generate_frame_labels(frame, channels)
            # Save
            fdir = self.writer.get_frame_dir(sc_name, w_name, frame.frame_idx)
            self.writer.write_json(os.path.join(fdir, "poses.json"), frame.agent_poses)
            self.writer.write_json(os.path.join(fdir, "labels.json"), labels.to_dict())

            # Save sensor data periodically
            if i % self.config.sensor_save_interval == 0:
                for aid, readings in frame.sensor_readings.items():
                    for sid, data in readings.items():
                        self.writer.write_sensor_data(fdir, aid, sid, data)

            # Save channel matrices
            for ch in channels:
                H = ch.get_frequency_response(self.config.communication)
                meta = {"tx_id": ch.tx_id, "rx_id": ch.rx_id,
                        "distance_m": ch.distance_m, "num_paths": ch.num_paths,
                        "tx_position": ch.tx_position.tolist(),
                        "rx_position": ch.rx_position.tolist()}
                self.writer.write_channel(fdir, ch.tx_id, ch.rx_id, H, meta)

            if (i+1) % max(1, n_frames//5) == 0:
                logger.info(f"  Processed {i+1}/{n_frames} frames")

        elapsed = time.time() - t0
        summary = {
            "scenario": sc_name, "town": self.config.scenario.town.value,
            "weather": w_name, "num_frames": len(frames),
            "num_agents": len(agents),
            "links_per_frame": len(channels) if frames else 0,
            "total_channels": total_channels,
            "carrier_freq_GHz": self.config.communication.carrier_frequency_hz/1e9,
            "subcarriers": self.config.communication.num_subcarriers,
            "Nt": self.config.communication.tx_num_antennas_2d,
            "Nr": self.config.communication.rx_num_antennas_2d,
            "codebook_Q": self.config.communication.codebook_size,
            "elapsed_s": round(elapsed, 1),
        }
        self.writer.write_json(
            os.path.join(self.config.output_dir, sc_name, w_name, "summary.json"),
            summary)
        self.config.save(os.path.join(self.config.output_dir, "config.json"))

        logger.info(f"\n{'='*65}")
        logger.info(f"COMPLETE in {elapsed:.1f}s — "
                    f"{total_channels} channels, {len(frames)} frames")
        logger.info(f"  Saved to: {self.config.output_dir}")
        logger.info("=" * 65)
        return summary


class BatchPipelineRunner:
    """Run pipeline across multiple scenarios × weather conditions."""
    def __init__(self, base_config, scenarios=None, weathers=None):
        self.base_config = base_config
        self.scenarios = scenarios or list(SCENARIO_LIBRARY.keys())[:3]
        self.weathers = weathers or [WeatherType.SUNNY, WeatherType.RAINY, WeatherType.FOGGY]

    def run_all(self):
        summaries = []
        for sc_name in self.scenarios:
            sc_cfg = SCENARIO_LIBRARY.get(sc_name)
            if sc_cfg is None: continue
            for weather in self.weathers:
                cfg = copy.deepcopy(self.base_config)
                cfg.scenario = copy.deepcopy(sc_cfg)
                cfg.scenario.weather = weather
                pipe = MultimodalWirelessPipeline(cfg)
                summaries.append(pipe.run())
        return summaries


# ╔════════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 12: MAIN — Generate dataset + visualize + prepare beam prediction    ║
# ╚════════════════════════════════════════════════════════════════════════════════╝

def print_hierarchy():
    print("""
┌────────────────────────────────────────────────────────────────────────┐
│                     OOD CLASS HIERARCHY                                │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  Sensor (ABC)                       Agent (ABC)                        │
│  ├── LiDARSensor                    ├── CAV  (mobile user / rx)        │
│  ├── RGBCameraSensor                └── RSU  (base station / tx)       │
│  ├── DepthCameraSensor                                                 │
│  ├── IMUSensor                      WeatherCondition (ABC)             │
│  └── RadarSensor                    ├── SunnyWeather                   │
│                                     ├── RainyWeather                   │
│  ScenarioGenerator (ABC)            └── FoggyWeather                   │
│  ├── CARLAScenarioGenerator                                           │
│  └── SyntheticScenarioGenerator     LabelGenerator (ABC)               │
│                                     ├── BeamLabelGenerator  (Eq.2-4)   │
│  SceneReconstructor (ABC)           ├── BoundingBoxLabelGenerator      │
│  ├── BlenderReconstructor           ├── ChannelStateLabelGenerator     │
│  └── SyntheticSceneBuilder          └── CompositeLabelGenerator        │
│                                                                        │
│  ChannelGenerator (ABC)             Pipeline Orchestrator              │
│  ├── SionnaChannelGenerator         ├── MultimodalWirelessPipeline     │
│  └── AnalyticalChannelGenerator     └── BatchPipelineRunner            │
│                                                                        │
│  Design Patterns:                                                      │
│  ─ Strategy    : swappable generators at every pipeline stage          │
│  ─ Factory     : SensorFactory, create_weather()                       │
│  ─ Composite   : CompositeLabelGenerator merges N label types          │
│  ─ Template    : Pipeline.run() fixed flow, pluggable stages           │
│  ─ Builder     : PipelineConfig with nested dataclasses                │
│  ─ Polymorphism: weather.degrade_lidar(), agent.get_role(), etc.       │
└────────────────────────────────────────────────────────────────────────┘
""")


def main():
    print_hierarchy()

    # ── Configure ──
    config = PipelineConfig(
        simulation=SimulationConfig(
            frame_rate_hz=100,
            scenario_duration_s=2.0,  # 2s demo (increase for full paper: 10s)
            random_seed=42,
        ),
        scenario=ScenarioConfig(
            town=TownID.TOWN03,
            scenario_name="roundabout",
            weather=WeatherType.SUNNY,
            num_cavs=3,
            description="Vehicles' movement in a roundabout",
        ),
        communication=CommunicationConfig(
            carrier_frequency_hz=28e9,
            subcarrier_spacing_hz=120e3,
            num_subcarriers=256,
            tx_num_antennas_2d=64,
            rx_num_antennas_2d=16,
            codebook_size=64,
        ),
        output_dir="./mmw_output",
    )

    print(f"Configuration:")
    print(f"  Scenario    : {config.scenario.scenario_name} in {config.scenario.town.value}")
    print(f"  Weather     : {config.scenario.weather.value}")
    print(f"  Frames      : {config.simulation.num_frames} @ {config.simulation.frame_rate_hz} Hz")
    print(f"  Carrier freq: {config.communication.carrier_frequency_hz/1e9:.1f} GHz")
    print(f"  Bandwidth   : {config.communication.bandwidth_hz/1e6:.1f} MHz")
    print(f"  TX/RX ant.  : {config.communication.tx_num_antennas_2d}/{config.communication.rx_num_antennas_2d}")
    print(f"  Codebook Q  : {config.communication.codebook_size}\n")

    # ── Run pipeline (sunny) ──
    pipeline = MultimodalWirelessPipeline(config)
    summary = pipeline.run()

    # ── Inspect generated data ──
    reader = DatasetReader("./mmw_output")
    sc, wt = config.scenario.scenario_name, config.scenario.weather.value

    print("\n" + "="*65)
    print("SAMPLE OUTPUT — Frame 0 Labels")
    print("="*65)
    labels_0 = reader.load_frame_labels(sc, wt, 0)
    if "beam" in labels_0:
        for link, bd in labels_0["beam"].items():
            print(f"\n  Link: {link}")
            print(f"    Optimal TX beam idx : {bd['optimal_tx_beam_idx']}")
            print(f"    Optimal RX beam idx : {bd['optimal_rx_beam_idx']}")
            print(f"    Max BF gain         : {bd['max_beamforming_gain']:.4e}")
            print(f"    Channel power (dB)  : {bd['channel_power_dB']:.1f}")
            print(f"    Distance (m)        : {bd['distance_m']:.1f}")
            print(f"    LOS                 : {bd['is_los']}")
            print(f"    Num paths           : {bd['num_paths']}")
    if "bounding_boxes" in labels_0:
        print(f"\n  Bounding boxes: {len(labels_0['bounding_boxes'])} vehicles")

    # ── Load channel matrix ──
    ch_dir = os.path.join("./mmw_output", sc, wt, "frames", "frame_000000", "channel")
    ch_files = [f for f in os.listdir(ch_dir) if f.endswith(".npz")]
    if ch_files:
        d = np.load(os.path.join(ch_dir, ch_files[0]))
        H = d["H_real"] + 1j * d["H_imag"]
        print(f"\n  Channel H shape: {H.shape}  [Nr, Nt, K]")

    # ── Generate multi-weather dataset (shorter for demo) ──
    print("\n" + "="*65)
    print("MULTI-WEATHER GENERATION (50 frames each)")
    print("="*65)
    for wtype in [WeatherType.RAINY, WeatherType.FOGGY]:
        cfg_w = copy.deepcopy(config)
        cfg_w.scenario.weather = wtype
        cfg_w.simulation.scenario_duration_s = 0.5  # 50 frames for quick demo
        pipe_w = MultimodalWirelessPipeline(cfg_w)
        pipe_w.run()

    # ── Build beam prediction training data ──
    print("\n" + "="*65)
    print("BEAM PREDICTION TRAINING DATA")
    print("="*65)
    n_frames = config.simulation.num_frames
    all_beams = []
    for i in range(n_frames):
        try:
            lbl = reader.load_frame_labels(sc, wt, i)
            if "beam" in lbl:
                first_link = list(lbl["beam"].keys())[0]
                all_beams.append(lbl["beam"][first_link]["optimal_tx_beam_idx"])
        except Exception: pass

    all_beams = np.array(all_beams)
    P, W = 40, 10   # Paper: P=40 history, W=10 prediction
    n_samples = len(all_beams) - P - W + 1

    if n_samples > 0:
        X = np.zeros((n_samples, P), dtype=np.int64)
        Y = np.zeros((n_samples, W), dtype=np.int64)
        for i in range(n_samples):
            X[i] = all_beams[i:i+P]; Y[i] = all_beams[i+P:i+P+W]
        np.savez("./mmw_output/beam_prediction_data.npz",
                 X_beam=X, Y_beam=Y, beam_sequence=all_beams)
        print(f"  History window P = {P} ({P*10}ms)")
        print(f"  Predict window W = {W} ({W*10}ms)")
        print(f"  Training samples = {n_samples}")
        print(f"  X_beam shape     = {X.shape}")
        print(f"  Y_beam shape     = {Y.shape}")
        print(f"  ✓ Saved beam_prediction_data.npz")
    else:
        print(f"  Need ≥{P+W} frames for beam prediction. Increase scenario_duration_s.")

    # ── Visualizations ──
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        # Plot 1: Channel power across subcarriers
        if ch_files:
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            pw = 10*np.log10(np.mean(np.abs(H)**2, axis=(0,1)) + 1e-30)
            axes[0].plot(pw); axes[0].set_xlabel("Subcarrier k")
            axes[0].set_ylabel("Power (dB)"); axes[0].set_title("Channel Power vs k")
            axes[0].grid(True, alpha=0.3)

            H_avg = np.mean(H, axis=2)
            im = axes[1].imshow(np.abs(H_avg), aspect='auto', cmap='viridis')
            axes[1].set_xlabel("TX Ant"); axes[1].set_ylabel("RX Ant")
            axes[1].set_title("|H̄| (avg subcarriers)"); plt.colorbar(im, ax=axes[1])

            im2 = axes[2].imshow(np.angle(H_avg), aspect='auto', cmap='twilight')
            axes[2].set_xlabel("TX Ant"); axes[2].set_ylabel("RX Ant")
            axes[2].set_title("∠H̄ (phase)"); plt.colorbar(im2, ax=axes[2])

            plt.tight_layout()
            plt.savefig("./mmw_output/channel_visualization.png", dpi=150)
            print("\n  ✓ Saved channel_visualization.png")

        # Plot 2: Beam tracking over time
        if len(all_beams) > 10:
            fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
            t_ms = np.arange(len(all_beams)) * 10

            gains, dists = [], []
            for i in range(len(all_beams)):
                try:
                    lbl = reader.load_frame_labels(sc, wt, i)
                    fl = list(lbl["beam"].keys())[0]
                    gains.append(lbl["beam"][fl]["channel_power_dB"])
                    dists.append(lbl["beam"][fl]["distance_m"])
                except: gains.append(0); dists.append(0)

            axes[0].plot(t_ms, all_beams, 'b.-', ms=2, lw=0.8)
            axes[0].set_ylabel("Optimal TX Beam q*"); axes[0].set_title("Beam Tracking")
            axes[0].set_ylim([-1, config.communication.codebook_size+1])
            axes[0].grid(True, alpha=0.3)

            axes[1].plot(t_ms, gains, 'g.-', ms=2, lw=0.8)
            axes[1].set_ylabel("Channel Power (dB)"); axes[1].grid(True, alpha=0.3)

            axes[2].plot(t_ms, dists, 'r.-', ms=2, lw=0.8)
            axes[2].set_ylabel("TX-RX Distance (m)"); axes[2].set_xlabel("Time (ms)")
            axes[2].grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig("./mmw_output/beam_tracking.png", dpi=150)
            print("  ✓ Saved beam_tracking.png")

        # Plot 3: DFT beam patterns
        Nt = config.communication.tx_num_antennas_2d
        Q = config.communication.codebook_size
        F = dft_codebook(Nt, Q)
        q_star = labels_0["beam"][list(labels_0["beam"].keys())[0]]["optimal_tx_beam_idx"]

        fig, ax = plt.subplots(figsize=(12, 5))
        theta = np.linspace(-np.pi/2, np.pi/2, 720)
        a_resp = np.exp(1j * np.pi * np.arange(Nt)[:, None] * np.sin(theta[None, :]))
        for q in range(Q):
            gain_db = 10*np.log10(np.abs(F[q] @ a_resp)**2 / Nt + 1e-10)
            if q == q_star:
                ax.plot(np.degrees(theta), gain_db, 'r-', lw=2.5,
                        label=f"Optimal q*={q_star}", zorder=10)
            else:
                ax.plot(np.degrees(theta), gain_db, 'b-', lw=0.3, alpha=0.12)
        ax.set_xlabel("Angle (°)"); ax.set_ylabel("Gain (dB)")
        ax.set_title(f"DFT Codebook (Nt={Nt}, Q={Q})"); ax.set_ylim([-30, 2])
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig("./mmw_output/beam_patterns.png", dpi=150)
        print("  ✓ Saved beam_patterns.png")

    except ImportError:
        print("\n  matplotlib not installed — skipping plots")

    # ── Dataset statistics ──
    total = sum(os.path.getsize(os.path.join(dp, f))
                for dp, _, fns in os.walk("./mmw_output") for f in fns)
    exts = {}
    for dp, _, fns in os.walk("./mmw_output"):
        for f in fns:
            e = os.path.splitext(f)[1]
            exts[e] = exts.get(e, 0) + 1

    print(f"\n{'='*65}")
    print(f"DATASET STATISTICS")
    print(f"{'='*65}")
    print(f"  Total size : {total/1024/1024:.1f} MB")
    print(f"  Location   : ./mmw_output/")
    print(f"  File types :")
    for e, c in sorted(exts.items(), key=lambda x: -x[1]):
        print(f"    {e or 'other':8s} : {c:6d} files")

    print(f"\n{'='*65}")
    print(f"ALL DONE — Dataset ready for downstream tasks")
    print(f"{'='*65}")
    print(f"\nTo scale up (full paper config), increase:")
    print(f"  scenario_duration_s = 10.0   (→ 1000 frames/scenario)")
    print(f"  num_subcarriers     = 1024   (→ full 120 MHz bandwidth)")
    print(f"  Run BatchPipelineRunner for 16 scenarios × 3 weathers")


if __name__ == "__main__":
    main()

18:46:43 [INFO] =================================================================
18:46:43 [INFO] MULTIMODAL-WIRELESS DATASET GENERATION
18:46:43 [INFO]   Scenario : roundabout | Town: Town03
18:46:43 [INFO]   Weather  : sunny | Frames: 200 @ 100 Hz
18:46:43 [INFO]   Antennas : Nt=64, Nr=16, Q=64
18:46:43 [INFO] =================================================================
18:46:43 [INFO] 
 STAGE 1: Scenario Execution & Data Capture
18:46:43 [INFO] Synthetic: 3 CAVs + 1 RSU, weather=sunny
18:46:43 [INFO]   Frame 40/200
18:46:43 [INFO]   Frame 80/200



┌────────────────────────────────────────────────────────────────────────┐
│                     OOD CLASS HIERARCHY                                │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│  Sensor (ABC)                       Agent (ABC)                        │
│  ├── LiDARSensor                    ├── CAV  (mobile user / rx)        │
│  ├── RGBCameraSensor                └── RSU  (base station / tx)       │
│  ├── DepthCameraSensor                                                 │
│  ├── IMUSensor                      WeatherCondition (ABC)             │
│  └── RadarSensor                    ├── SunnyWeather                   │
│                                     ├── RainyWeather                   │
│  ScenarioGenerator (ABC)            └── FoggyWeather                   │
│  ├── CARLAScenarioGenerator                                           │
│  └── SyntheticScenarioG

18:46:43 [INFO]   Frame 120/200
18:46:43 [INFO]   Frame 160/200
18:46:43 [INFO]   Frame 200/200
18:46:43 [INFO] Synthetic scenario complete
18:46:43 [INFO]    200 frames, 4 agents
18:46:43 [INFO] 
 STAGES 2-4: Reconstruct → Channel → Labels → Save
18:46:52 [INFO]   Processed 40/200 frames
18:47:01 [INFO]   Processed 80/200 frames
18:47:09 [INFO]   Processed 120/200 frames
18:47:18 [INFO]   Processed 160/200 frames
18:47:27 [INFO]   Processed 200/200 frames
18:47:27 [INFO] 
18:47:27 [INFO] COMPLETE in 43.7s — 600 channels, 200 frames
18:47:27 [INFO]   Saved to: ./mmw_output
18:47:27 [INFO] =================================================================
18:47:27 [INFO] =================================================================
18:47:27 [INFO] MULTIMODAL-WIRELESS DATASET GENERATION
18:47:27 [INFO]   Scenario : roundabout | Town: Town03
18:47:27 [INFO]   Weather  : rainy | Frames: 50 @ 100 Hz
18:47:27 [INFO]   Antennas : Nt=64, Nr=16, Q=64
18:47:27 [INFO] =========================


SAMPLE OUTPUT — Frame 0 Labels

  Link: rsu_00_to_cav_00
    Optimal TX beam idx : 5
    Optimal RX beam idx : 59
    Max BF gain         : 4.1482e-10
    Channel power (dB)  : -90.4
    Distance (m)        : 37.9
    LOS                 : True
    Num paths           : 8

  Link: rsu_00_to_cav_01
    Optimal TX beam idx : 21
    Optimal RX beam idx : 26
    Max BF gain         : 3.2815e-11
    Channel power (dB)  : -94.4
    Distance (m)        : 59.2
    LOS                 : True
    Num paths           : 8

  Link: rsu_00_to_cav_02
    Optimal TX beam idx : 41
    Optimal RX beam idx : 23
    Max BF gain         : 7.9352e-11
    Channel power (dB)  : -91.6
    Distance (m)        : 38.3
    LOS                 : True
    Num paths           : 8

  Bounding boxes: 3 vehicles

  Channel H shape: (16, 64, 256)  [Nr, Nt, K]

MULTI-WEATHER GENERATION (50 frames each)


18:47:27 [INFO]   Frame 20/50
18:47:27 [INFO]   Frame 30/50
18:47:27 [INFO]   Frame 40/50
18:47:27 [INFO]   Frame 50/50
18:47:27 [INFO] Synthetic scenario complete
18:47:27 [INFO]    50 frames, 4 agents
18:47:27 [INFO] 
 STAGES 2-4: Reconstruct → Channel → Labels → Save
18:47:29 [INFO]   Processed 10/50 frames
18:47:32 [INFO]   Processed 20/50 frames
18:47:34 [INFO]   Processed 30/50 frames
18:47:36 [INFO]   Processed 40/50 frames
18:47:38 [INFO]   Processed 50/50 frames
18:47:38 [INFO] 
18:47:38 [INFO] COMPLETE in 11.6s — 150 channels, 50 frames
18:47:38 [INFO]   Saved to: ./mmw_output
18:47:38 [INFO] =================================================================
18:47:38 [INFO] =================================================================
18:47:38 [INFO] MULTIMODAL-WIRELESS DATASET GENERATION
18:47:38 [INFO]   Scenario : roundabout | Town: Town03
18:47:38 [INFO]   Weather  : foggy | Frames: 50 @ 100 Hz
18:47:38 [INFO]   Antennas : Nt=64, Nr=16, Q=64
18:47:38 [INFO] ===========


BEAM PREDICTION TRAINING DATA
  History window P = 40 (400ms)
  Predict window W = 10 (100ms)
  Training samples = 151
  X_beam shape     = (151, 40)
  Y_beam shape     = (151, 10)
  ✓ Saved beam_prediction_data.npz

  ✓ Saved channel_visualization.png
  ✓ Saved beam_tracking.png
  ✓ Saved beam_patterns.png

DATASET STATISTICS
  Total size : 1691.8 MB
  Location   : ./mmw_output/
  File types :
    .json    :   1894 files
    .npy     :    930 files
    .npz     :    901 files
    .png     :      3 files

ALL DONE — Dataset ready for downstream tasks

To scale up (full paper config), increase:
  scenario_duration_s = 10.0   (→ 1000 frames/scenario)
  num_subcarriers     = 1024   (→ full 120 MHz bandwidth)
  Run BatchPipelineRunner for 16 scenarios × 3 weathers
